### Resume Screening Assistant (LangChain + RAG)

In [60]:
import os
import glob

from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.runnables import RunnablePassthrough

In [61]:
# load API Key
load_dotenv()

True

In [62]:
# Model to be used for LLM and Embeddings

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [63]:
# Load the resume PDF

resume_files = sorted(glob.glob("sample_resumes/*.pdf"))
resume_files

['sample_resumes\\Resume_A_Priya_Nair.pdf',
 'sample_resumes\\Resume_B_Arjun_Mehta.pdf',
 'sample_resumes\\Resume_C_Sneha_Rao.pdf']

In [64]:
#Load one sample PDF to check what comes out

loader = PyPDFLoader(resume_files[0])
documents = loader.load()

print("Pages:", len(documents))
print(documents[0].page_content[:400])

Pages: 1
Priya Nair
Data Scientist | Bengaluru, India | priya.nair@email.com | +91 98XXX XXXX
Professional Summary
Data Scientist with 5 years of experience building and deploying machine learning models for retail and fintech clients.
Strong background in statistics, experimentation and production ML.
Technical Skills
Python, SQL, Pandas, NumPy, Scikit-learn, XGBoost, TensorFlow, PyTorch, Statistics, Hypo


In [65]:
#Chunking, Embeddings and FAIS per resume to avoid put all resumes into a single index, a search for the job description pulls the best chunks across *all* candidates, so a weak resume gets evaluated partly on a strong candidate's content. 

In [66]:
# Function to builds one vector store per resume

def build_vectorstore(file_path, candidate_name):

    # Load the PDF
    loader = PyPDFLoader(file_path)
    documents = loader.load()

    # Chunking (same settings as the class RAG project)
    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    docs = splitter.split_documents(documents)

    # Tag every chunk with the candidate name
    for d in docs:
        d.metadata["candidate"] = candidate_name

    # Embeddings + FAISS
    vectorstore = FAISS.from_documents(docs, embeddings)

    return vectorstore, docs

In [67]:
# Build the vector store for every resume

resume_stores = {}   # {candidate_name: FAISS store}
all_docs = []        # every chunk of every resume

for file_path in resume_files:

    candidate_name = os.path.basename(file_path).replace(".pdf", "")

    vs, docs = build_vectorstore(file_path, candidate_name)

    resume_stores[candidate_name] = vs
    all_docs.extend(docs)

    print(candidate_name, "->", len(docs), "chunks")

Resume_A_Priya_Nair -> 2 chunks
Resume_B_Arjun_Mehta -> 2 chunks
Resume_C_Sneha_Rao -> 1 chunks


In [68]:
# One combined store for the question answering part

combined_store = FAISS.from_documents(all_docs, embeddings)

print("Combined store ready with", len(all_docs), "chunks")

Combined store ready with 5 chunks


In [69]:
# Job Description

job_description = """
Role: Data Scientist

Must have:
Python, SQL, Machine Learning, Scikit-learn, Statistics, Pandas,
Data Visualization, Model Evaluation

Good to have:
Deep Learning, NLP, LangChain, AWS, MLOps, A/B Testing

Experience:
3+ years of experience building and deploying machine learning models.
"""

print(job_description)


Role: Data Scientist

Must have:
Python, SQL, Machine Learning, Scikit-learn, Statistics, Pandas,
Data Visualization, Model Evaluation

Good to have:
Deep Learning, NLP, LangChain, AWS, MLOps, A/B Testing

Experience:
3+ years of experience building and deploying machine learning models.



In [70]:
# Evaluation prompt

evaluation_prompt = ChatPromptTemplate.from_template("""
You are an experienced technical recruiter.

Evaluate the candidate ONLY using the resume context given below.
If something is not present in the context, treat it as missing.
Do NOT invent any experience, skill or qualification.

JOB DESCRIPTION:
{jd}

RESUME CONTEXT:
{context}

Return ONLY valid JSON in exactly this format (no extra text):
{{
 "candidate_name": "",
 "match_score": 0,
 "matching_skills": [],
 "missing_skills": [],
 "candidate_summary": "",
 "strengths": [],
 "weaknesses": [],
 "recommendation": "",
 "justification": ""
}}

Rules:
- "match_score" is a number between 0 and 100.
- "recommendation" must be one of: "Strong Fit", "Moderate Fit", "Not a Fit".
- Keep every list item short (maximum 8 words).
- "justification" must be 2 to 3 lines explaining the score.
""")

In [71]:
# Helper + evaluation function

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


def evaluate_resume(vectorstore, job_description, candidate_name):

    # Retriever pulls the 4 chunks closest to the JD
    retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
    docs = retriever.invoke(job_description)
    context = format_docs(docs)

    # Chain: prompt -> llm -> JSON parser
    evaluation_chain = evaluation_prompt | llm | JsonOutputParser()

    result = evaluation_chain.invoke({
        "jd": job_description,
        "context": context
    })

    result["candidate_name"] = candidate_name

    return result

In [72]:
# Evaluate the first resume

candidate_a = list(resume_stores.keys())[0]

result_a = evaluate_resume(resume_stores[candidate_a], job_description, candidate_a)

result_a

{'candidate_name': 'Resume_A_Priya_Nair',
 'match_score': 90,
 'matching_skills': ['Python',
  'SQL',
  'Machine Learning',
  'Scikit-learn',
  'Statistics',
  'Pandas',
  'Data Visualization',
  'Model Evaluation',
  'A/B Testing'],
 'missing_skills': ['Deep Learning', 'NLP', 'MLOps'],
 'candidate_summary': 'Data Scientist with 5 years experience.',
 'strengths': ['Strong background in statistics',
  'Experience with A/B testing',
  'AWS SageMaker expertise',
  'Proficient in Python and SQL'],
 'weaknesses': ['Missing deep learning experience', 'No mention of MLOps'],
 'recommendation': 'Strong Fit',
 'justification': 'The candidate meets most required skills and has relevant experience. Missing some good-to-have skills but overall a strong match.'}

In [73]:
# Print it in a readable way

def show_result(r):
    print("Candidate      :", r["candidate_name"])
    print("Match Score    :", r["match_score"], "/ 100")
    print("Recommendation :", r["recommendation"])
    print("\nMatching Skills:", ", ".join(r["matching_skills"]))
    print("Missing Skills :", ", ".join(r["missing_skills"]))
    print("\nStrengths      :", ", ".join(r["strengths"]))
    print("Weaknesses     :", ", ".join(r["weaknesses"]))
    print("\nSummary        :", r["candidate_summary"])
    print("\nJustification  :", r["justification"])


show_result(result_a)

Candidate      : Resume_A_Priya_Nair
Match Score    : 90 / 100
Recommendation : Strong Fit

Matching Skills: Python, SQL, Machine Learning, Scikit-learn, Statistics, Pandas, Data Visualization, Model Evaluation, A/B Testing
Missing Skills : Deep Learning, NLP, MLOps

Strengths      : Strong background in statistics, Experience with A/B testing, AWS SageMaker expertise, Proficient in Python and SQL
Weaknesses     : Missing deep learning experience, No mention of MLOps

Summary        : Data Scientist with 5 years experience.

Justification  : The candidate meets most required skills and has relevant experience. Missing some good-to-have skills but overall a strong match.


In [74]:
# Evaluate all other candidates

evaluations = {}

for name, store in resume_stores.items():
    evaluations[name] = evaluate_resume(store, job_description, name)
    print("Done:", name, "->", evaluations[name]["match_score"])

Done: Resume_A_Priya_Nair -> 90
Done: Resume_B_Arjun_Mehta -> 85
Done: Resume_C_Sneha_Rao -> 30


In [75]:
# Score table

for name, r in sorted(evaluations.items(), key=lambda x: -x[1]["match_score"]):
    print(f"{r['match_score']:>3} | {name:<25} | {r['recommendation']}")

 90 | Resume_A_Priya_Nair       | Strong Fit
 85 | Resume_B_Arjun_Mehta      | Moderate Fit
 30 | Resume_C_Sneha_Rao        | Not a Fit


In [76]:
# Missing skills for every candidate

for name, r in evaluations.items():
    print(name)
    for skill in r["missing_skills"]:
        print("   -", skill)
    print()

Resume_A_Priya_Nair
   - Deep Learning
   - NLP
   - MLOps

Resume_B_Arjun_Mehta
   - Statistics
   - Pandas
   - Data Visualization
   - Model Evaluation
   - Deep Learning
   - NLP
   - LangChain
   - A/B Testing

Resume_C_Sneha_Rao
   - Python
   - Machine Learning
   - Scikit-learn
   - Statistics
   - Pandas
   - Data Visualization
   - Model Evaluation
   - Deep Learning
   - NLP
   - LangChain
   - AWS
   - MLOps
   - A/B Testing



In [77]:
#18. Comparison chain & Recommendation -> plain text answer

comparison_prompt = ChatPromptTemplate.from_template("""
You are a hiring manager.

Below are the evaluation results of all candidates for the same job.

JOB DESCRIPTION:
{jd}

CANDIDATE EVALUATIONS:
{evaluations}

Give your answer in this format:
1. Best Candidate: <name>
2. Why this candidate is the best: 3 bullet points
3. Ranking of all candidates with their scores
4. Final hiring recommendation in 2 lines
""")

comparison_chain = comparison_prompt | llm | StrOutputParser()

answer = comparison_chain.invoke({
    "jd": job_description,
    "evaluations": str(evaluations)
})

print(answer)

1. Best Candidate: Priya Nair

2. Why this candidate is the best:
   - Priya has 5 years of experience in data science, exceeding the required 3+ years, which demonstrates her depth of knowledge and expertise.
   - She possesses all the must-have skills for the role, including strong statistical background and experience with A/B testing and AWS deployment.
   - Despite missing some good-to-have skills, her overall match score of 90 and relevant experience make her a strong fit for the position.

3. Ranking of all candidates with their scores:
   - Resume_A_Priya_Nair: 90
   - Resume_B_Arjun_Mehta: 85
   - Resume_C_Sneha_Rao: 30

4. Final hiring recommendation: Hire Priya Nair as she is the strongest candidate with the required skills and experience. Consider providing training for the good-to-have skills she is missing.


In [78]:
#RAG chain over all resumes to answer questions about candidates from  the uploaded resumes

qa_prompt = ChatPromptTemplate.from_template("""
You are a resume screening assistant.

Answer the question using ONLY the resume context below.
If the answer is not present in the context, reply exactly:
"This information is not available in the uploaded resumes."

Context:
{context}

Question: {question}

Answer in short bullet points.
""")

retriever = combined_store.as_retriever(search_kwargs={"k": 4})

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | qa_prompt
    | llm
    | StrOutputParser()
)

In [79]:
# test a sample question that IS answerable from the resumes

print(rag_chain.invoke("Which candidate has Kubernetes experience?"))

- Arjun Mehta has Kubernetes experience.
- Deployed machine learning models as REST APIs using FastAPI and Docker on Kubernetes.


In [80]:
#test a question that is NOT in the resumes to prove there is no hallucination

print(rag_chain.invoke("Which candidate speaks French?"))

This information is not available in the uploaded resumes.


In [81]:
# The above  same logic is put in `resume_engine.py` and used by the Streamlit app in `app.py`.
# Pipeline: PDF Loader -> Text Splitter -> Embeddings -> FAISS -> Retriever -> Prompt -> LLM -> Output Parser

